<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Estimación del modelo de crecimiento exportador

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `statsmodels`: estimación por mínimos cuadrados y modelos lineales generalizados.
- `pickle`: para serializar el modelo ajustado.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import pickle

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import temp, tables, model

### Lectura y transformación de variables

Carga la base de episodios y construye las transformaciones logarítmicas y cuadráticas que
entran a la regresión. Excluye a Colombia de la muestra de estimación, porque el modelo se
estima con la experiencia de terceros países y luego se aplica al caso colombiano.

In [3]:
df = pd.read_parquet(temp / 'exportaciones' / 'df_para_estimacion.parquet')

df['log_Delta_x'] = np.log1p(df['Delta_x'])
df['log_Delta_m'] = np.sign(df['Delta_m']) * np.log1p(np.abs(df['Delta_m']))

df['log_distancia'] = np.log1p(df['distancia'])

df['log_gdp_o'] = np.log1p(df['gdp_o'])
df['log_gdp_d'] = np.log1p(df['gdp_d'])

df['log_gdp_o2'] = df['log_gdp_o'] ** 2
df['log_gdp_d2'] = df['log_gdp_d'] ** 2

df['log_duracion'] = np.log1p(df['Duración'])
df['log_duracion2'] = df['log_duracion'] ** 2

df.query("o_iso3 != 'COL'", inplace=True)
df.dropna(inplace=True) # Faltan gdp's en paises raros

### Variables del modelo

Define la variable dependiente y el conjunto de regresores: el cambio en la demanda del destino,
la participación inicial, la duración del episodio, el ingreso de origen y destino con sus
términos cuadráticos, la distancia, la similitud cultural y los efectos fijos de subregión e
industria.

In [4]:
Y = df['log_Delta_x']
X = df[['log_Delta_m', 'share_start',
        'log_duracion', 'log_duracion2', 
        'log_gdp_o', 'log_gdp_o2', 
        'log_gdp_d', 'log_gdp_d2', 
        'subregion_o', 'subregion_d', 'cultura_similaridad', 'log_distancia','industria']]

### Construcción de la matriz de diseño

Convierte las variables categóricas en indicadoras, descartando una categoría por grupo para
evitar colinealidad, y añade la constante.

In [5]:
# Convertir variables categóricas a dummies
X = pd.get_dummies(X, columns=['subregion_o', 'subregion_d', 'industria'], drop_first=True)

# Añadir constante a X
X = sm.add_constant(X)

### Estimación por mínimos cuadrados

Ajusta la especificación lineal en logaritmos con errores robustos a heterocedasticidad.

In [6]:
# Ajustar el modelo OLS con errores robustos tipo HC3
model_hc3 = sm.OLS(Y, X.astype(float)).fit(cov_type="HC3")

# Imprimir el resumen del modelo
print(model_hc3.summary())

                            OLS Regression Results                            
Dep. Variable:            log_Delta_x   R-squared:                       0.405
Model:                            OLS   Adj. R-squared:                  0.405
Method:                 Least Squares   F-statistic:                     5152.
Date:                Sun, 20 Sep 2026   Prob (F-statistic):               0.00
Time:                        07:42:00   Log-Likelihood:            -6.4921e+05
No. Observations:              353741   AIC:                         1.299e+06
Df Residuals:                  353694   BIC:                         1.299e+06
Df Model:                          46                                         
Covariance Type:                  HC3                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

### Estimación del modelo lineal generalizado

Ajusta la especificación de Poisson con vínculo logarítmico sobre el nivel del cambio en
exportaciones, con errores robustos. Es la que se usa en la proyección.

In [7]:
Y = df['Delta_x']

# Ajustar el modelo GLM con errores robustos tipo HC3
model_glm = sm.GLM(Y, X.astype(float), family=sm.families.Poisson(link=sm.families.links.Log())).fit(cov_type="HC3")

# Imprimir el resumen del modelo
print(model_glm.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                Delta_x   No. Observations:               353741
Model:                            GLM   Df Residuals:                   353694
Model Family:                 Poisson   Df Model:                           46
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.3501e+10
Date:                Sun, 20 Sep 2026   Deviance:                   2.6998e+10
Time:                        07:42:04   Pearson chi2:                 4.62e+11
No. Iterations:                    11   Pseudo R-squ. (CS):              1.000
Covariance Type:                  HC3                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

### Persistencia del modelo y de la base de estimación

Guarda el modelo ajustado y la base con la que se estimó. El modelo serializado es lo que
consume la proyección y la tabla de coeficientes del deck.

In [8]:
model.mkdir(parents=True, exist_ok=True)
with open(model / 'glm_model.pkl', 'wb') as f:
    pickle.dump(model_glm, f)

pd.concat([Y, X], axis=1).to_parquet(temp / 'exportaciones' / 'df_regresion_glm.parquet')

### Tabla de coeficientes para la presentación

Exporta los coeficientes estimados en formato LaTeX. En el proyecto original este paso vivía en
un script suelto, `leer_coeficientes_regresion.py`, fuera de los notebooks; aquí se integra a la
cadena porque la presentación consume su salida.

In [9]:
tables.mkdir(parents=True, exist_ok=True)

df_coeficientes = model_glm.params.to_frame(name='Coeficiente')
df_coeficientes.index.name = 'Variable'
df_coeficientes.to_latex(tables / 'coeficientes_regresion.tex', float_format="{:.4f}".format)

print(df_coeficientes.head(12).to_string())

                          Coeficiente
Variable                             
const                           -2.93
log_Delta_m                      0.11
share_start                      1.86
log_duracion                     2.08
log_duracion2                   -0.22
log_gdp_o                        0.09
log_gdp_o2                      -0.00
log_gdp_d                        0.68
log_gdp_d2                      -0.01
cultura_similaridad             -0.01
log_distancia                   -0.31
subregion_o_Eastern Asia        -0.37
